In [1]:
import pandas as pd
import numpy as np
import pickle
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

df = pd.read_csv("C:\\Users\Admin\\Car_recommendation_system\\processed_data(cars24).csv")
print(df.shape)

(1000, 12)


In [2]:
df['year'] = df['name'].str.split().str[0].astype(int)

df['tags'] = (
    df['name'] + " " +
    df['variant'] + " " +
    df['fuel type'] + " " +
    df['transmission'] + " " +
    df['registration']
)

df['tags'] = df['tags'].str.lower()

In [3]:
cv = CountVectorizer(max_features=5000, stop_words='english')
text_vectors = cv.fit_transform(df['tags']).toarray()

print("Vector Shape:", text_vectors.shape)

numeric_features = df[['price','km driven','year']]

scaler = StandardScaler()
numeric_scaled = scaler.fit_transform(numeric_features)

combined_features = np.hstack((text_vectors, numeric_scaled))

similarity = cosine_similarity(combined_features)
print("Similarity Shape:", similarity.shape)

Vector Shape: (1000, 85)
Similarity Shape: (1000, 1000)


In [4]:
 def recommend(car_name):

    car_index = df[df['name'] == car_name].index[0]

    distances = similarity[car_index]

    car_list = sorted(
        list(enumerate(distances)),
        reverse=True,
        key=lambda x: x[1]
    )[1:20]

    recommended = []

    base_price = df.iloc[car_index]['price']

    for i in car_list:

        car = df.iloc[i[0]]

        if abs(car['price'] - base_price) < 300000:

            recommended.append(car)

        if len(recommended) == 5:
            break

    print("\nRecommended Cars:\n")

    for car in recommended:
        print(car['name'])

recommend(df['name'].iloc[0])


Recommended Cars:

2012 Maruti Alto 800
2012 Maruti Alto 800
2012 Maruti Alto 800
2012 Maruti Alto 800
2012 Maruti Alto 800


In [5]:
pickle.dump(df, open("C:\\Users\Admin\\Car_recommendation_system\\car_data.pkl","wb"))
pickle.dump(similarity, open("C:\\Users\Admin\\Car_recommendation_system\\similarity.pkl","wb"))